# Silver Layer Processing

## Phase 3: Silver Layer Transformation

In this notebook, we are processing validated NYC Taxi datasets and generating clean Silver layer Delta tables.

Objectives:
- recreate validation logic
- identify valid and invalid records
- generate clean datasets
- store validated Delta tables
- prepare data for analytical workloads

The Silver layer contains cleaned, validated, and transformation-ready datasets.

#### Step 1: Import required libraries 

In [0]:
from pyspark.sql.functions import *

#### Step 2: Load Raw Dataset from Bronze Layer

In [0]:
df = spark.read.parquet(
    "/Volumes/taxi_quality_platform/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet"
)

#### Step 3: Recreate Validation Rules

In [0]:
# fare_amount, trip_distance, passenger_count, payment_type, timestamp_validation_status

validated_df = df\
    .withColumn(
        "fare_validation_status", 
            when(
                (col("fare_amount").isNull()) |
                (col("fare_amount") <= 0),
                "FAILED"
            ).otherwise("PASSED")           
    )\
    .withColumn(
        "distance_validation_status",
        when(
            (col("trip_distance").isNull()) |
            (col("trip_distance") <= 0),
            "FAILED"
        ).otherwise("PASSED")
    )\
    .withColumn(
        "passenger_validation_status",
        when(
            (col("passenger_count").isNull()) |
            (col("passenger_count") <= 0),
            "FAILED"
        ).otherwise("PASSED")
    )\
    .withColumn(
        "payment_validation_status",
        when(
            (col("payment_type").isNull()),
            "FAILED"
        ).otherwise("PASSED")
        
    )\
    .withColumn(
        "timestamp_validation_status",
        when(
            (col("tpep_pickup_datetime") > col("tpep_dropoff_datetime")),
            "FAILED"
        ).otherwise("PASSED")
        
    )


#### Step 4: Generate Overall Validation Status

In [0]:
# create a column for overall validation

validated_df = validated_df.withColumn(
    "overall_validation_status",
    when(
        (col("fare_validation_status") == "FAILED") |
        (col("distance_validation_status") == "FAILED") |
        (col("passenger_validation_status") == "FAILED") |
        (col("payment_validation_status") == "FAILED") |
        (col("timestamp_validation_status") == "FAILED"),
        "FAILED"
    ).otherwise("PASSED")
)


#### Step 5: Separate Valid and Invalid Records

In [0]:
# We are separating:
# - valid records for Silver tables
# - invalid records for monitoring and audit analysis

valid_records_df = validated_df.filter(col("overall_validation_status") == "PASSED")
invalid_records_df = validated_df.filter(col("overall_validation_status") == "FAILED")

#### Step 6: Remove Duplicate Records

In [0]:
valid_records_df = valid_records_df.dropDuplicates()

#### Step 7: Standardize Dataset Columns

In [0]:
# Change the datatypes for the columns fare_amount, passenger_count, trip_distance

valid_records_df = valid_records_df\
    .withColumn(
        "fare_amount", 
        col("fare_amount").cast("double")
    )\
    .withColumn(
        "trip_distance", 
        col("trip_distance").cast("double")
    )\
    .withColumn(
        "passenger_count", 
        col("passenger_count").cast("integer")
    )

#### Step 8: Store Validated Silver Layer Table

In [0]:
valid_records_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("taxi_quality_platform.silver.validated_taxi_trips")

#### Step 9: Store Failed Validation Records

In [0]:
# Invalid records are stored separately for:
# - audit analysis
# - quality monitoring
# - anomaly tracking

invalid_records_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("taxi_quality_platform.silver.invalid_taxi_trips")

#### Step 10: Verify Silver Layer Tables

In [0]:
silver_df = spark.read.table("taxi_quality_platform.silver.validated_taxi_trips")

silver_df.show(5)
                             

In [0]:
print("total silver records: ", silver_df.count())

In [0]:
invalid_silver_df = spark.read.table("taxi_quality_platform.silver.invalid_taxi_trips")

print("total invalid silver records:", invalid_silver_df.count())

# Silver Layer Processing Summary

In this notebook, we:
- processed validated datasets
- separated valid and invalid records
- generated clean Silver layer Delta tables
- stored failed validation records for monitoring
- prepared datasets for Gold layer analytics

The Silver layer now contains trusted and analytics-ready datasets.